In [1]:
from screenshot import get_screen_with_grid
import cv2

In [2]:
images = get_screen_with_grid()

overlayed = images[1]
original  = images[0]

In [3]:
cv2.imshow("overlayed_image", overlayed)
k = cv2.waitKey(0)

# 4. Close all OpenCV windows
cv2.destroyAllWindows()

In [ ]:
from PIL import Image
from google import genai
from google.genai import types

from dotenv import load_dotenv

load_dotenv()

client = genai.Client()

image = Image.fromarray(overlayed)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[image, "If you were to be instructed to open edge browser by the user of the computer shown in the image, how would you go about it? Your response should be strictly a json output. "
    "It should look like this: {'action': 'action goes here' , 'target': 'the cell to be targeted', 'value':'the value to type or the key to click'}. In case of moving the mouse, you would output something like this: "
    "{'action':'click', 'target':'C5'}. In case you also wanted to type, the value would be this: {'action':'click_and_type', 'target':'C5', 'value':'Hello World'}.  "],
    config=types.GenerateContentConfig(
        temperature=1.0
    )
)
print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 54.509942846s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

In [4]:
import re

# main_text = "Some intro text. {Desired block of text}. {More text follows.}"
pattern = r"{(.*?)}"

matches = re.findall(pattern, response.text)
instructions = []


for i in matches:
    # Use .strip() to remove spaces and quotes from both the key and the value
    my_dict = {
        key.strip(" '\""): value.strip(" '\"") 
        for item in i.split(',') 
        for key, value in [item.split(':')]
    }
    instructions.append(my_dict)
    # Output: Desired block of text


In [5]:
instructions

[{'action': 'click', 'target': 'AL1'}]

In [6]:
def label_to_coords(label, step_size=50):
    chars = re.findall(r'[A-Za-z]+|\d+', label)

    # print(chars)
    
    letter_part = chars[0].upper()
    number = int(chars[1])

    # Convert standard Excel-style column letters to a number (A=1, Z=26, AA=27)
    letter_index = 0
    for char in letter_part:
        # Multiply current total by 26 and add the character's value (A=1...Z=26)
        letter_index = letter_index * 26 + (ord(char) - 64)

    # print(f"Letter string: {letter_part}, Calculated Index: {letter_index}")

    x = (letter_index * step_size) - (step_size // 2)
    y = (number * step_size) - (step_size // 2)

    return x, y

In [7]:
label_to_coords('AA1')

(1325, 25)

In [9]:
action_to_take = instructions[0]

In [10]:
action_to_take

{'action': 'click', 'target': 'AL1'}

In [11]:
target = label_to_coords(action_to_take['target'])
action = action_to_take['action']
# value = action_to_take['value'] | None



In [ ]:
import pyautogui


if action == 'click':

    pyautogui.moveTo(target)
    pyautogui.click()


elif action == 'click_and_type':
    pyautogui.moveTo(target)
    pyautogui.click()
    pyautogui.typewrite(value)